# Hermite Timing Wheel: BAO Acoustic Modes

**Ainulindale Engine** — Tier 8

H_n has n real zeros = n BAO timing marks. γ_n/(2π) = BK scale x₀(n). CMB peaks = H_n levels.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import importlib.util, math, cmath, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

def load_maths(tier):
    paths = {7: '../../modules/tier7_cosmos/maths.py',
             8: '../../modules/tier8_sedenion/maths.py',
             9: '../../modules/tier9_chem/maths.py'}
    spec = importlib.util.spec_from_file_location(f't{tier}', paths[tier])
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

OMEGA_ZS = 0.5671432904097838
D_STAR   = 0.24600
GAP      = OMEGA_ZS - D_STAR * math.log(10)
R_H      = 1.0 / math.sqrt(2.0)
RIEMANN_ZEROS = [14.134725, 21.022040, 25.010858, 30.424876, 32.935062,
                 37.586178, 40.918719, 43.327073, 48.005151, 49.773832]
print("Ainulindale notebook ready.")


In [ ]:
m = load_maths(8)
result = m.hermite_timing_wheel()
print("Claim:", result['claim'])
print("Confidence:", result['confidence'])


## Engine Output

In [ ]:
import pprint
pprint.pprint({k: v for k, v in result.items()
               if k not in ('claim', 'confidence', 'latex')}, width=100, depth=3)


## Mathematical Foundation

$$\text{LaTeX: }\quad H_n\text{ zeros}=n\text{ BAO modes},\;x_0(n)=\gamma_n/2\pi$$

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def eval_hermite(n, x):
    if n==0: return 1.0
    if n==1: return 2.0*x
    h0,h1 = 1.0, 2.0*x
    for k in range(2,n+1): h2=2*x*h1-2*(k-1)*h0; h0,h1=h1,h2
    return h1

# Left: H_n for n=0..5
x_arr = np.linspace(-3, 3, 400)
ax = axes[0]
for n in range(6):
    y = [eval_hermite(n,x) for x in x_arr]
    ax.plot(x_arr, np.clip(y,-15,15), label=f'H_{n}', alpha=0.8)
ax.axhline(0, color='k', lw=0.8); ax.set_ylim(-15,15)
ax.set_xlabel('x'); ax.set_title('Hermite Polynomials H_n(x)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Middle: timing wheel (zeros as n-gon)
ax2 = axes[1]
for n in [3,5,7]:
    # Find zeros approximately
    x_max = math.sqrt(2*n+4)
    xs = np.linspace(-x_max,x_max,2000)
    ys = np.array([eval_hermite(n,x) for x in xs])
    zeros = []
    for i in range(len(ys)-1):
        if ys[i]*ys[i+1]<0:
            a,b = float(xs[i]),float(xs[i+1])
            for _ in range(40):
                m=(a+b)/2
                if eval_hermite(n,m)*eval_hermite(n,a)<0: b=m
                else: a=m
            zeros.append((a+b)/2)
    scale = math.sqrt(2*n+1)
    scaled = [z/scale for z in zeros]
    # Place on unit circle
    theta_vals = [math.pi*s for s in scaled]
    ax2.scatter([math.cos(t) for t in theta_vals], [math.sin(t) for t in theta_vals],
                s=60, label=f'n={n} ({len(zeros)} zeros)', zorder=5)
theta_circle = np.linspace(0,2*math.pi,300)
ax2.plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', alpha=0.3)
ax2.set_aspect('equal'); ax2.set_title('Hermite Zeros as BAO Timing Marks')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# Right: Riemann zeros → BK levels
ax3 = axes[2]
x0_vals = [g/(2*math.pi) for g in RIEMANN_ZEROS]
ax3.scatter(range(1,11), RIEMANN_ZEROS, c='steelblue', s=80, zorder=5, label='γ_n (Riemann zero)')
ax3.plot(range(1,11), RIEMANN_ZEROS, 'steelblue', alpha=0.5)
ax3r = ax3.twinx()
ax3r.scatter(range(1,11), x0_vals, c='red', s=80, marker='s', zorder=5, label='x₀(n)=γ_n/2π')
ax3r.plot(range(1,11), x0_vals, 'r--', alpha=0.5)
ax3.set_xlabel('n'); ax3.set_ylabel('γ_n', color='steelblue'); ax3r.set_ylabel('x₀(n)', color='red')
ax3.set_title('BK Mapping: γ_n/(2π) = Hermite Scale')
ax3.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Ainulindale Reading

**Confidence:** `{}'.format(result.get('confidence',''))`

Run `result.keys()` to explore the full engine output.

In [ ]:
print('Engine keys:', list(result.keys()))